In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. TẢI VÀ TIỀN XỬ LÝ DỮ LIỆU CƠ BẢN
# ==========================================
print("Đang đọc dữ liệu...")
# Lưu ý: Thay đổi đường dẫn file cho phù hợp với máy của bạn
df = pd.read_csv('jena_climate_2009_2016.csv')

# Chuyển đổi cột 'Date Time' sang định dạng Datetime của Pandas và đặt làm Index
df['Date Time'] = pd.to_datetime(df['Date Time'], format='%d.%.m.%Y %H:%M:%S')
df.set_index('Date Time', inplace=True)

# Lấy ra cột Nhiệt độ (T (degC)) để phân tích chính
temp = df['T (degC)']

# Thiết lập phong cách đồ thị
sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.figsize': (15, 6), 'font.size': 12})

# ==========================================
# 2. TÍNH CHU KỲ VÀ MÙA VỤ (SEASONALITY)
# ==========================================
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# a. Chu kỳ ngày (Zoom vào 3 ngày đầu tiên: 3 ngày * 24 giờ * 6 (10phút/lần) = 432 dòng)
axes[0].plot(temp.iloc[:432], color='orange')
axes[0].set_title('Chu kỳ ngày: Biến thiên nhiệt độ trong 3 ngày đầu tiên (Cứ 10 phút / lần)')
axes[0].set_ylabel('Nhiệt độ (°C)')

# b. Chu kỳ năm (Zoom out toàn bộ dữ liệu, lấy giá trị trung bình theo ngày để đồ thị bớt rối)
daily_temp = temp.resample('D').mean() # Downsampling: Trung bình theo ngày (D)
axes[1].plot(daily_temp, color='teal', alpha=0.8)
axes[1].set_title('Chu kỳ năm (Tính mùa vụ): Nhiệt độ trung bình ngày từ 2009 - 2016')
axes[1].set_ylabel('Nhiệt độ (°C)')

plt.tight_layout()
plt.show()

# ==========================================
# 3. TÍNH TƯƠNG QUAN ĐA BIẾN (CORRELATION HEATMAP)
# ==========================================
plt.figure(figsize=(12, 10))
# Tính ma trận tương quan (Pearson)
corr_matrix = df.corr()

# Vẽ Heatmap
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Bản đồ nhiệt (Heatmap): Mức độ tương quan giữa các biến thời tiết', fontsize=16)
plt.tight_layout()
plt.show()

# ==========================================
# 4. TÍNH XU HƯỚNG (TREND) BẰNG ĐƯỜNG TRUNG BÌNH ĐỘNG
# ==========================================
plt.figure(figsize=(15, 6))

# Lấy dữ liệu trung bình tháng để nhìn rõ xu hướng dài hạn
monthly_temp = temp.resample('M').mean()

# Tính đường trung bình động (Moving Average) chu kỳ 12 tháng (1 năm)
moving_avg_12m = monthly_temp.rolling(window=12, center=True).mean()

plt.plot(monthly_temp, label='Nhiệt độ trung bình tháng', color='lightgray', marker='o')
plt.plot(moving_avg_12m, label='Xu hướng (Moving Average 12 tháng)', color='red', linewidth=3)
plt.title('Phân tích Xu hướng (Trend): Liệu nhiệt độ có đang tăng dần?')
plt.ylabel('Nhiệt độ (°C)')
plt.legend()
plt.tight_layout()
plt.show()

# ==========================================
# 5. TÌM NHIỄU VÀ DỮ LIỆU NGOẠI LAI (ANOMALIES)
# ==========================================
plt.figure(figsize=(15, 6))

# Vẽ biểu đồ hộp (Boxplot) để dễ dàng nhìn thấy các điểm ngoại lai (Outliers) theo từng tháng
df['Month'] = df.index.month
sns.boxplot(x='Month', y='T (degC)', data=df, palette='Set3')
plt.title('Phân bổ nhiệt độ theo tháng (Boxplot) - Các chấm đen nằm ngoài râu là dữ liệu bất thường')
plt.xlabel('Tháng')
plt.ylabel('Nhiệt độ (°C)')
plt.tight_layout()
plt.show()

# Xóa cột Month tạm thời nếu không cần dùng nữa
df.drop(columns=['Month'], inplace=True)